#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### data

In [ ]:
# set dataset parameters
dataset = 'synthetic'
train_size = 1000

In [ ]:
# set confounders
confounders = ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']
input_dim = len(confounders)

In [ ]:
# read
data_path = ROOT / "data" / "datasets" / f"{dataset}.csv"
df = pd.read_csv(data_path, index_col=0)

#### helpers

In [ ]:
def load_nuisance_models(ns_seed_dir, input_dim, device):
    """Helper to load nuisance models. Set the hidden dim correctly!"""
    paths = {
        "e": ns_seed_dir / "prop_model.pt",
        "m0": ns_seed_dir / "mu0_model.pt",
        "m1": ns_seed_dir / "mu1_model.pt"}

    for name, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing nuisance checkpoint {name}: {path}")

    prop_model = ClassificationHead(input_dim=input_dim, hidden_dim=128).to(device)
    prop_model.load_state_dict(torch.load(paths["e"], map_location=device, weights_only=True))

    m0_model = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
    m0_model.load_state_dict(torch.load(paths["m0"], map_location=device, weights_only=True))

    m1_model = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
    m1_model.load_state_dict(torch.load(paths["m1"], map_location=device, weights_only=True))

    return prop_model, m0_model, m1_model

In [ ]:
class DRRankerDataset(Dataset):    
    def __init__(self, df, confounders, kappa):
        # plug-in predictions
        df = df.copy()
        df["tau_hat"] = df["m1_hat"] - df["m0_hat"]

        # tensors
        self.X     = torch.as_tensor(df[confounders].to_numpy(), dtype=torch.float32)
        self.tau   = torch.as_tensor(df["tau_hat"].to_numpy(), dtype=torch.float32)
        self.DR    = torch.as_tensor(df["DR"].to_numpy(), dtype=torch.float32)

        # other
        self.N = self.X.size(0)
        self.kappa = float(kappa)

    def __len__(self):
        return self.N * (self.N - 1)

    
    @staticmethod
    def k_to_ij(k: int, N: int):
        """
        k is index in [0, N*(N-1)], (i, j) is pair with i != j.
        """
        i = k // (N - 1)
        j = k % (N - 1)
        if j >= i:
            j += 1
        return i, j

    def __getitem__(self, idx):
        i, j = self.k_to_ij(idx, self.N)

        # features
        x_i = self.X[i]
        x_j = self.X[j]

        # soft label with DR targets
        soft = torch.sigmoid((self.DR[i] - self.DR[j]) / self.kappa)
        
        # return DR based label twice so we can reuse training code
        return x_i, x_j, soft, soft

In [ ]:
def make_DR_ranker_loaders(train_df, val_df, confounders, kappa, batch_size):
    train_ds = DRRankerDataset(train_df, confounders, kappa)
    val_ds = DRRankerDataset(val_df, confounders, kappa)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/DR_ranker.csv', index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    hidden_dim=int(row["hidden_dim"]),
    learning_rate=float(row["lr"]),
    weight_decay=float(row["weight_decay"]),
    batch_size=int(row["batch_size"]),
    kappa=float(row["kappa"]),
    max_epochs=50,
    patience=5)

In [ ]:
# set directories
out_dir = f'./chkpts/{dataset}/'
os.makedirs(out_dir, exist_ok=True)

ns_dir = ROOT / "experiments" / "nuisances" / "chkpts" / dataset

In [ ]:
# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}, size {train_size}")
    set_seed(seed)

    # set output dir
    ckpt_dir = Path(out_dir) / f"size_{train_size}" / f"seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # get training data
    _, _, train_df, val_df, _ = make_splits(df=df, train_size=train_size, seed=seed)

    # load nuisance models
    ns_seed_dir = ns_dir / f"size_{train_size}" / f"seed_{seed}"
    prop_model, m0_model, m1_model = load_nuisance_models(ns_seed_dir=ns_seed_dir, input_dim=input_dim, device=device)

    # add pseudo outcomes to dataframes
    train_df = compute_dr_scores(train_df, confounders, prop_model, m0_model, m1_model, device)
    val_df = compute_dr_scores(val_df, confounders, prop_model, m0_model, m1_model, device)

    # make data loaders
    train_loader, _ = make_DR_ranker_loaders(train_df, val_df, confounders, params['kappa'], params['batch_size'])
    _, val_loader = make_cate_loaders(train_df, val_df, confounders, params['batch_size'])

    # init model
    ranker = ClassificationHead(input_dim, hidden_dim=params['hidden_dim']).to(device)
    
    # train model
    ranker, info = train_ranker(ranker, train_loader, val_loader, device, lr=params['learning_rate'], weight_decay=params['weight_decay'],
                                      max_epochs=50, patience=5, seed=seed, fraction_of_pairs=0.1, plug_in=True)

    # checkpoint
    torch.save(ranker.state_dict(), ckpt_dir / f"DR_ranker.pt")

#### evaluation

In [ ]:
def load_DR_ranker(train_size, params, seed, confounders, device):
    input_dim = len(confounders)

    # set checkpoint path
    ckpt_path = ROOT / "experiments" / "supplementary" / "baselines" / "DR_ranker" / "chkpts" / f"size_{train_size}" / f"seed_{seed}" / "DR_ranker.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

    # load model checkpoint
    model = ClassificationHead(input_dim, hidden_dim=params['hidden_dim']).to(device)
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    model.eval()
    
    return model

In [ ]:
def get_estimates_DR_ranker(model, test_loader, device):
    # put model to eval
    model.eval()
    
    # collectors
    DR_hat, cate_true, M0_true, M1_true = [], [], [], []
    
    # loop over test loader
    with torch.inference_mode():
        for x, cate, M0, M1 in test_loader:
            
            # put tensorts on device
            x = x.to(device)
            preds = model(x).squeeze(-1)  # shape [B]
            
            # collect
            DR_hat.append(preds)
            cate_true.append(cate.squeeze(-1))
            M0_true.append(M0.squeeze(-1))
            M1_true.append(M1.squeeze(-1))
    
    # concat -> cpu -> numpy
    to_np = lambda parts: torch.cat(parts, dim=0).detach().cpu().numpy()
    
    df_eval = pd.DataFrame({
        "DR_ranker_score": to_np(DR_hat),     
        "cate":    to_np(cate_true),    
        "M0":      to_np(M0_true),
        "M1":      to_np(M1_true)})

    return df_eval

In [ ]:
def compute_metrics_DR_ranker(eval_df):
    required = {"DR_ranker_score", "cate"}

    # check if estimates are available
    missing = required - set(eval_df.columns)
    if missing:
        raise ValueError(f"eval_df is missing required columns: {sorted(missing)}")

    specs = [("DR_ranker", "DR_ranker_score")]
    rows = []
    for name, score_col in specs:
        ranked = eval_df.sort_values(score_col, ascending=False).copy()

        rows.append({
            "model": name,
            "autoc": autoc(ranked),
            "policy_value": policy_value(ranked)})

    return pd.DataFrame(rows)

In [ ]:
# init collector
all_metrics = []

# loop over seeds and sizes
for seed in range(5):
    for size in [100, 250, 500, 1000, 2000]:

        # track progress
        print(f" -> Seed {seed}, size {size}")
        set_seed(seed)

        # get testing data
        _, _, _, _, test_df = make_splits(df=df, train_size=size, seed=seed)
        test_loader = DataLoader(EvalDataset(test_df, confounders), batch_size=1024, shuffle=False)

        # load model
        model = load_DR_ranker(size, params, seed, confounders, device)
        df_eval = get_estimates_DR_ranker(model, test_loader, device)
        df_metrics = compute_metrics_DR_ranker(df_eval)

        # store
        df_metrics["size"] = size
        df_metrics["seed"] = seed
        all_metrics.append(df_metrics)

# summarize
df_all = pd.concat(all_metrics, ignore_index=True)
summary = (df_all.groupby(["size", 'model'])[["autoc", "policy_value"]].agg(['mean', 'std']).reset_index())